# Milestone 5
Master the competition metric, optimise inference, and combine models for maximum performance. Introduce TTA.

Suggested Readings:

- [PyTorch – Softmax Function](https://docs.pytorch.org/docs/2.13/generated/torch.nn.functional.softmax.html)
- [Hugging Face – Transformer](https://huggingface.co/docs/transformers/index)
- [Ensemble Learning](https://scikit-learn.org/stable/modules/ensemble.html)

Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

Setup (Run Before Attempting Questions) :

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

In [1]:
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

df=pd.read_csv("../../data/raw/train.csv")

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

DeBERTa_tokenizer=AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
DeBERTa=AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-small", num_labels=5).to(device)

RoBERTa_tokenizer=AutoTokenizer.from_pretrained("roberta-base")
RoBERTa=AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5).to(device)

LABEL_MAP={"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
LABEL_MAP_REV={v: k for k, v in LABEL_MAP.items()}

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?


(answer format : eg - A, probability of A)

In [12]:
def build_input(row, tokenizer):
    prompt=row['prompt']

    options="\n".join(
        f"{label}: {row[label]}"
        for label in LABEL_MAP.keys()
    )

    text=f"Question: {prompt}\n\n{options}"

    tokenized_inputs=tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)

    return tokenized_inputs

def get_probabilities(row, model, tokenizer, input_builder=build_input):
    inputs=input_builder(row, tokenizer)
    inputs={k: v.to(device) for k, v in inputs.items()} # move to device
    with torch.no_grad():
        output=model(**inputs)
    logits=output.logits.squeeze(0)
    probs=logits.softmax(dim=0)
    return probs.cpu() #back to CPU

#DeBERTa on row 25
prob_DeBERTa_row25=get_probabilities(df.iloc[25], DeBERTa, DeBERTa_tokenizer)

for i, p in enumerate(prob_DeBERTa_row25):
    print(f"{LABEL_MAP_REV[i]}, {p.item():.4f}")
print("Highest probability option:", LABEL_MAP_REV[prob_DeBERTa_row25.argmax().item()],",", prob_DeBERTa_row25.max().item())

A, 0.1665
B, 0.1876
C, 0.2412
D, 0.1061
E, 0.2986
Highest probability option: E , 0.298583984375


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [6]:
prob_RoBERTa_row25=get_probabilities(df.iloc[25], RoBERTa, RoBERTa_tokenizer)

avg_prob_row_25=(prob_DeBERTa_row25+prob_RoBERTa_row25)/2
print("Highest probability option:", LABEL_MAP_REV[avg_prob_row_25.argmax().item()],",", avg_prob_row_25.max().item())

Highest probability option: E , 0.27569180727005005


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [7]:
weighted_avg=(prob_DeBERTa_row25*0.7 + prob_RoBERTa_row25*0.3)

print("Highest probability option:", LABEL_MAP_REV[weighted_avg.argmax().item()],",", weighted_avg.max().item())

Highest probability option: E , 0.2848242521286011


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E

In [8]:
def get_top_3_labels_form_prob(probabilities):
    top3_idx=torch.argsort(probabilities, descending=True)[:3]
    top3_options=" ".join([LABEL_MAP_REV[i.item()] for i in top3_idx])
    return top3_options

print("Top 3 options:", get_top_3_labels_form_prob(weighted_avg))

Top 3 options: E C B


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [17]:
test_df=pd.read_csv("../../data/raw/test.csv")

predictions=[]

# inference
for _, row in test_df.iterrows():
    prob_DeBERTa=get_probabilities(row, DeBERTa, DeBERTa_tokenizer)
    prob_RoBERTa=get_probabilities(row, RoBERTa, RoBERTa_tokenizer)
    weighted_avg=(prob_DeBERTa*0.7 + prob_RoBERTa*0.3)
    top_3_options=get_top_3_labels_form_prob(weighted_avg)
    predictions.append((row['id'], top_3_options))

submission_df=pd.DataFrame(predictions, columns=["id", "prediction"])
print(len(predictions))

500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [18]:
def modified_build_input(row, tokenizer):
    prompt=row['prompt']

    options="\n".join(
        f"{label}: {row[label]}"
        for label in LABEL_MAP.keys()
    )

    text=f"Answer the following multiple-choice question carefully:\nQuestion: {prompt}\n\n{options}"

    tokenized_inputs=tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)

    return tokenized_inputs

original_probs=[]
modified_probs=[]

for _, row in test_df[:50].iterrows():
    original_probs.append(get_probabilities(row, DeBERTa, DeBERTa_tokenizer, input_builder=build_input))
    modified_probs.append(get_probabilities(row, DeBERTa, DeBERTa_tokenizer, input_builder=modified_build_input))

top_1_label_original=[LABEL_MAP_REV[torch.argmax(p).item()] for p in original_probs]
top_1_label_modified=[LABEL_MAP_REV[torch.argmax(p).item()] for p in modified_probs]

# no of different predictions
count=0
for o, m in zip(top_1_label_original, top_1_label_modified):
    if o != m:
        count+=1

print(count)

32


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

In [22]:
count=0
for _,row in test_df[:100].iterrows():
    deberta_probs=get_probabilities(row, DeBERTa, DeBERTa_tokenizer)
    roberta_probs=get_probabilities(row, RoBERTa, RoBERTa_tokenizer)
    weighted_probs=(deberta_probs*0.7 + roberta_probs*0.3)

    top_1_deberta=LABEL_MAP_REV[torch.argmax(deberta_probs).item()]
    top_1_weighted=LABEL_MAP_REV[torch.argmax(weighted_probs).item()]
    if top_1_deberta != top_1_weighted:
        count+=1
print(count)


23


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [24]:
confidence_gains=[]
for _, row in test_df[:100].iterrows():
    deberta_probs=get_probabilities(row, DeBERTa, DeBERTa_tokenizer)
    roberta_probs=get_probabilities(row, RoBERTa, RoBERTa_tokenizer)
    weighted_probs=(deberta_probs*0.7 + roberta_probs*0.3)

    top_1_prob_deberta=deberta_probs.max().item()
    top_1_prob_weighted=weighted_probs.max().item()

    confidence_gain=top_1_prob_weighted - top_1_prob_deberta
    confidence_gains.append(confidence_gain)

print(len([x for x in confidence_gains if x > 0]))

5


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [25]:
count=0
for _, row in test_df[:100].iterrows():
    deberta_probs=get_probabilities(row, DeBERTa, DeBERTa_tokenizer)
    roberta_probs=get_probabilities(row, RoBERTa, RoBERTa_tokenizer)
    weighted_probs=(deberta_probs*0.7 + roberta_probs*0.3)

    top_3_options_deberta=get_top_3_labels_form_prob(deberta_probs)
    top_3_options_weighted=get_top_3_labels_form_prob(weighted_probs)

    # no of rows with atlease one change in top 3 options
    if set(top_3_options_deberta.split()) != set(top_3_options_weighted.split()):
        count+=1

print(count)

40


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score?

(Round to 4 decimal places.)

In [27]:
def map3(prediction:list, ground_truth:str) -> float:
    if not prediction:
        return 0.0
    score = 0.0
    for i, pred in enumerate(prediction[:3]):
        if pred == ground_truth:
            score += 1 / (i + 1)
    return score

score=0
for _, row in df[:100].iterrows():
    deberta_probs=get_probabilities(row, DeBERTa, DeBERTa_tokenizer)
    roberta_probs=get_probabilities(row, RoBERTa, RoBERTa_tokenizer)
    weighted_probs=(deberta_probs*0.7 + roberta_probs*0.3)

    top_3_options_weighted=get_top_3_labels_form_prob(deberta_probs)
    map3_score=map3(top_3_options_weighted.split(), row['answer'])
    score+=map3_score
score/=100
print(round(score, 4))

0.375
